In [122]:
import numpy as np
import pandas as pd
import os
import sys
import seaborn as sns
import pickle
import matplotlib.pyplot as plt
import scipy.stats as stats 
import missingno as msno
import sklearn as sk
import optuna
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.impute import KNNImputer
from sklearn.preprocessing import PolynomialFeatures, RobustScaler, StandardScaler, PowerTransformer
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from feature_engine.outliers import Winsorizer
sys.path.append("../src/Preprocesamiento")
from custom_transformers import OutlierHandler
import matplotlib.pyplot as plt
import plotly
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [123]:
SEMILLA = 42
FILENAME = "dataset_final.csv"
datapath = os.path.join("..","data","processed",FILENAME)
df = pd.read_csv(datapath)

In [124]:
y = df['target_Frio_kW_next']
df = df.drop(columns=['target_Frio_kW_next'])

In [125]:
with open ("../src/Preprocesamiento/preprocesamiento.pkl", "rb") as f:
    preprocessor = pickle.load(f)

In [126]:
df_preprocesado = preprocessor.transform(df)


### Separación de datos

In [127]:
X_train, X_test, y_train, y_test = train_test_split(df, y, train_size=0.7, shuffle=False, random_state=SEMILLA)
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, train_size=0.5, shuffle=False, random_state=SEMILLA)


In [128]:
X_train = preprocessor.transform(X_train)
print("train: ", X_train.shape)
X_val = preprocessor.transform(X_val)
print("val: ", X_val.shape)
X_test = preprocessor.transform(X_test)
print("test: ", X_test.shape)

train:  (805, 18)
val:  (172, 18)
test:  (173, 18)


### Baselines: MAE

Día de hoy

In [129]:
print(type(preprocessor.named_steps['Escalado']))

<class 'sklearn.compose._column_transformer.ColumnTransformer'>


In [130]:
columns_no_ciclicas = X_train.columns[~X_train.columns.isin(["cos_month_of_year", "sin_month_of_year"])]
df_sin_escalar = (
    preprocessor.named_steps['Escalado']
    .named_transformers_['Escalado']
    .set_output(transform="pandas")
    .inverse_transform(X_val[columns_no_ciclicas])
)
df_sin_escalar = pd.DataFrame(df_sin_escalar, columns=columns_no_ciclicas)

In [131]:
y_val

805    16144.0
806    12401.0
807    15177.0
808    17596.0
809    17489.0
        ...   
972    20378.0
973    21941.0
974    22115.0
975    17634.0
976    22816.0
Name: target_Frio_kW_next, Length: 172, dtype: float64

In [132]:
y_frio_hoy = df_sin_escalar['Frio (Kw)'].values
MAE_frio_hoy = np.mean(np.abs(y_frio_hoy - y_val))
print(f"MAE entre Frio (Kw) y target_Frio_kW_next: {MAE_frio_hoy}")

MAE entre Frio (Kw) y target_Frio_kW_next: 2724.4767441860463


Promedio móvil

In [133]:
y_prom_movil = df_sin_escalar['Prom movil Frio'].values
MAE_prom_movil = np.mean(np.abs(y_prom_movil - y_val))
print(f"MAE entre Prom movil Frio y target_Frio_kW_next: {MAE_prom_movil:.4f}")

MAE entre Prom movil Frio y target_Frio_kW_next: 2393.5963


## Entrenamiento de modelos

### Gradient Boosting

In [135]:
GBoost = GradientBoostingRegressor(loss="absolute_error" , random_state=SEMILLA)
GBoost.fit(X_train, y_train)
y_pred = GBoost.predict(X_val)
MAE_gxboost = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_gxboost:.4f}")

MAE: 2153.3883


### Random Forest Regressor

In [136]:
randomforest = RandomForestRegressor(criterion="absolute_error",random_state=SEMILLA)
randomforest.fit(X_train, y_train)
MAE_randomforest = np.mean(np.abs(randomforest.predict(X_val) - y_val))
print(f"MAE: {MAE_randomforest:.4f}")

MAE: 2493.6965


### lightGBM

In [137]:
lightgbm = LGBMRegressor(objective="mae",random_state=SEMILLA,verbose=-1)
lightgbm.fit(X_train, y_train)
y_pred = lightgbm.predict(X_val)
MAE_lightgbm = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_lightgbm:.4f}")

MAE: 2245.0937


### Lasso

In [138]:
lasso = Lasso(random_state=SEMILLA)
lasso.fit(X_train, y_train)
y_pred = lasso.predict(X_val)
MAE_lasso = np.mean(np.abs(y_pred - y_val))
print(f"MAE: {MAE_lasso:.4f}")

MAE: 2366.2171


### Busqueda de hiperparámetros

In [139]:
import json
def log_experiment(modelo,parametros,metricas,seed=42,path="results/experiment_logs.csv"):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path):
        df_existing = pd.read_csv(path)
        if not df_existing.empty:
            next_id = df_existing['experiment_id'].max() +1
        else:
            next_id = 0
    else:
        next_id = 0
    
    data = {'experiment_id': next_id,
            'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'seed': seed,
            'model': modelo,
            'params': json.dumps(parametros),
            **metricas}
    
    fila = pd.DataFrame([data])
    
    if os.path.exists(path):
        fila.to_csv(path,mode='a',header=False,index=False)
    else:
        fila.to_csv(path,mode='w',header=True,index=False)
    
    
    

### Gradient Boosting

In [140]:
def objective_gxb(trial):
    
   n_estimators = trial.suggest_int("n_estimators", 50,500)
   max_depth = trial.suggest_int("max_depth", 1, 5)

   learning_rate = trial.suggest_float("learning_rate", 0.005, 0.15, log=True)
   
   
   model = GradientBoostingRegressor(
       loss="absolute_error",
       n_estimators= n_estimators,
       max_depth= max_depth,
       learning_rate= learning_rate,
       random_state=SEMILLA
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   mae = mean_absolute_error(y_pred,y_val)
   rmse = np.sqrt(mean_squared_error(y_pred,y_val))
   R2 = r2_score(y_pred,y_val)
   log_experiment(modelo="GradientBoostingRegressor",
                  parametros = trial.params,
                  metricas= {"MAE": mae, "RMSE": rmse, "R2": R2})
   score = mae
   return score




sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction = "minimize", sampler=sampler)
study.optimize(objective_gxb, n_trials=100, n_jobs=1)

[I 2026-11-14 23:29:37,815] A new study created in memory with name: no-name-80df1f75-06f0-432c-bb25-79b658513805
[I 2026-11-14 23:29:39,784] Trial 0 finished with value: 2185.2486637016987 and parameters: {'n_estimators': 218, 'max_depth': 5, 'learning_rate': 0.060285631437218844}. Best is trial 0 with value: 2185.2486637016987.
[I 2026-11-14 23:29:40,449] Trial 1 finished with value: 2474.385414626029 and parameters: {'n_estimators': 319, 'max_depth': 1, 'learning_rate': 0.008499489191350382}. Best is trial 0 with value: 2185.2486637016987.
[I 2026-11-14 23:29:41,171] Trial 2 finished with value: 2433.741009061364 and parameters: {'n_estimators': 76, 'max_depth': 5, 'learning_rate': 0.038626891946536776}. Best is trial 0 with value: 2185.2486637016987.
[I 2026-11-14 23:29:41,896] Trial 3 finished with value: 2303.7342075208103 and parameters: {'n_estimators': 369, 'max_depth': 1, 'learning_rate': 0.13540804321249839}. Best is trial 0 with value: 2185.2486637016987.
[I 2026-11-14 23:2

In [141]:
import optuna.visualization as vis
import plotly

In [142]:

fig_history = vis.plot_optimization_history(study)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study)
fig_history.show()
fig_contour = vis.plot_contour(study)
fig_contour.show()


fig_importance = vis.plot_param_importances(study)
fig_importance.show()

### RandomForest

In [143]:
def objective_rf(trial):
   n_estimators = trial.suggest_int("n_estimators", 50, 300)
   max_depth = trial.suggest_int("max_depth", 5, 30)
   min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
   min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)

   model = RandomForestRegressor(
      criterion="absolute_error",
       n_estimators= n_estimators,
       max_depth= max_depth,
       min_samples_split= min_samples_split,
       min_samples_leaf= min_samples_leaf,
       random_state=SEMILLA
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   mae = mean_absolute_error(y_pred,y_val)
   rmse = np.sqrt(mean_squared_error(y_pred,y_val))
   R2 = r2_score(y_pred,y_val)
   log_experiment(modelo="RandomForestRegressor",
                  parametros = trial.params,
                  metricas= {"MAE": mae, "RMSE": rmse, "R2": R2})
   score = mae
   return score

study_rf = optuna.create_study(direction = "minimize", sampler=sampler)
study_rf.optimize(objective_rf, n_trials=100)

[I 2026-11-14 23:31:42,808] A new study created in memory with name: no-name-194604ab-21e8-46c4-ac07-45fa77d7dca0
[I 2026-11-14 23:31:46,620] Trial 0 finished with value: 2328.3458007137924 and parameters: {'n_estimators': 202, 'max_depth': 9, 'min_samples_split': 3, 'min_samples_leaf': 10}. Best is trial 0 with value: 2328.3458007137924.
[I 2026-11-14 23:31:53,059] Trial 1 finished with value: 2481.4228257406817 and parameters: {'n_estimators': 292, 'max_depth': 26, 'min_samples_split': 7, 'min_samples_leaf': 1}. Best is trial 0 with value: 2328.3458007137924.
[I 2026-11-14 23:31:56,492] Trial 2 finished with value: 2287.8549010838683 and parameters: {'n_estimators': 221, 'max_depth': 16, 'min_samples_split': 4, 'min_samples_leaf': 5}. Best is trial 2 with value: 2287.8549010838683.
[I 2026-11-14 23:31:57,278] Trial 3 finished with value: 2331.368584603047 and parameters: {'n_estimators': 58, 'max_depth': 28, 'min_samples_split': 6, 'min_samples_leaf': 7}. Best is trial 2 with value: 

In [144]:

fig_history = vis.plot_optimization_history(study_rf)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study_rf)
fig_history.show()
fig_contour = vis.plot_contour(study_rf)
fig_contour.show()


fig_importance = vis.plot_param_importances(study_rf)
fig_importance.show()

### lightGBM

In [145]:
def objective_lgbm(trial):
   n_estimators = trial.suggest_int("n_estimators", 50,200)
   max_depth = trial.suggest_int("max_depth", 3, 20)
   learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
   reg_alpha = trial.suggest_float("reg_alpha", 0.0, 0.1)
   reg_lambda = trial.suggest_float("reg_lambda", 0.0, 0.1)
   
   model = LGBMRegressor(
       n_estimators= n_estimators,
       max_depth= max_depth,
       learning_rate= learning_rate,
       reg_alpha= reg_alpha,
       reg_lambda= reg_lambda,
       random_state=SEMILLA,
       verbose=-1
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   mae = mean_absolute_error(y_pred,y_val)
   rmse = np.sqrt(mean_squared_error(y_pred,y_val))
   R2 = r2_score(y_pred,y_val)
   log_experiment(modelo="LGBMRegressor",
                  parametros = trial.params,
                  metricas= {"MAE": mae, "RMSE": rmse, "R2": R2})
   score = mae
   return score

study_lgbm = optuna.create_study(direction = "minimize", sampler=sampler)
study_lgbm.optimize(objective_lgbm, n_trials=100, n_jobs=1)

[I 2026-11-14 23:36:56,199] A new study created in memory with name: no-name-31e0b04d-a82a-40c5-859a-cc05c2691a99
[I 2026-11-14 23:36:56,475] Trial 0 finished with value: 2493.1204065564816 and parameters: {'n_estimators': 166, 'max_depth': 6, 'learning_rate': 0.011601413965844696, 'reg_alpha': 0.08154614284548342, 'reg_lambda': 0.07068573438476171}. Best is trial 0 with value: 2493.1204065564816.
[I 2026-11-14 23:36:56,773] Trial 1 finished with value: 2522.564449637533 and parameters: {'n_estimators': 160, 'max_depth': 16, 'learning_rate': 0.031472949002886205, 'reg_alpha': 0.035846572854427265, 'reg_lambda': 0.011586905952512973}. Best is trial 0 with value: 2493.1204065564816.
[I 2026-11-14 23:36:57,168] Trial 2 finished with value: 2622.5742677085964 and parameters: {'n_estimators': 180, 'max_depth': 14, 'learning_rate': 0.10596042720726825, 'reg_alpha': 0.006355835028602364, 'reg_lambda': 0.031098232171566223}. Best is trial 0 with value: 2493.1204065564816.
[I 2026-11-14 23:36:5

In [146]:

fig_history = vis.plot_optimization_history(study_lgbm)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study_lgbm)
fig_history.show()
fig_contour = vis.plot_contour(study_lgbm)
fig_contour.show()


fig_importance = vis.plot_param_importances(study_lgbm)
fig_importance.show()

### Lasso

In [147]:
def objective_lasso(trial):
   alpha = trial.suggest_float("alpha", 0.1, 5000, log=True)
   
   model = Lasso(
       alpha= alpha,
       random_state=SEMILLA,
   )

   model.fit(X_train, y_train)
   y_pred = model.predict(X_val)
   mae = mean_absolute_error(y_pred,y_val)
   rmse = np.sqrt(mean_squared_error(y_pred,y_val))
   R2 = r2_score(y_pred,y_val)
   log_experiment(modelo="Lasso",
                  parametros = trial.params,
                  metricas= {"MAE": mae, "RMSE": rmse, "R2": R2})
   score = mae
   return score

study_lasso = optuna.create_study(direction = "minimize")
study_lasso.optimize(objective_lasso, n_trials=100)

[I 2026-11-14 23:37:13,611] A new study created in memory with name: no-name-780211a5-42c5-41a0-9caf-7ae552c016d6
[I 2026-11-14 23:37:13,641] Trial 0 finished with value: 2350.494545596332 and parameters: {'alpha': 10.44425354535461}. Best is trial 0 with value: 2350.494545596332.
[I 2026-11-14 23:37:13,684] Trial 1 finished with value: 2306.527727255191 and parameters: {'alpha': 45.18625097550949}. Best is trial 1 with value: 2306.527727255191.
[I 2026-11-14 23:37:13,728] Trial 2 finished with value: 2362.89961297479 and parameters: {'alpha': 4.606122720838637}. Best is trial 1 with value: 2306.527727255191.
[I 2026-11-14 23:37:13,776] Trial 3 finished with value: 2365.878632806388 and parameters: {'alpha': 0.28919636731621834}. Best is trial 1 with value: 2306.527727255191.
[I 2026-11-14 23:37:13,809] Trial 4 finished with value: 2304.286552566366 and parameters: {'alpha': 47.41127999414742}. Best is trial 4 with value: 2304.286552566366.
[I 2026-11-14 23:37:13,842] Trial 5 finished 

In [148]:

fig_history = vis.plot_optimization_history(study_lasso)
fig_history.show()
fig_history = vis.plot_parallel_coordinate(study_lasso)
fig_history.show()
fig_contour = vis.plot_contour(study)
fig_contour.show()


fig_importance = vis.plot_param_importances(study)
fig_importance.show()

In [149]:
def objective_gxboost(trial):
    n_estimators = trial.suggest_int("n_estimators", 100, 800)
    max_depth = trial.suggest_int("max_depth", 3, 9)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.2, log=True)
    
    # Regularización y muestreo para evitar overfitting
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    subsample = trial.suggest_float("subsample", 0.6, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0)
    
    reg_alpha = trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True)

    model = XGBRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        min_child_weight=min_child_weight,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=SEMILLA,
        eval_metric="mae"   
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_pred,y_val)
    rmse = np.sqrt(mean_squared_error(y_pred,y_val))
    R2 = r2_score(y_pred,y_val)
    log_experiment(modelo="XGBRegressor",
                    parametros = trial.params,
                    metricas= {"MAE": mae, "RMSE": rmse, "R2": R2})
    score = mae
    return score

study_gxboost = optuna.create_study(direction="minimize")
study_gxboost.optimize(objective_gxboost, n_trials=100)

[I 2026-11-14 23:37:18,199] A new study created in memory with name: no-name-302831b0-f45b-4ea0-b780-2628f58ec607
[I 2026-11-14 23:37:18,608] Trial 0 finished with value: 2766.1486191860463 and parameters: {'n_estimators': 151, 'max_depth': 6, 'learning_rate': 0.0830945876073418, 'min_child_weight': 5, 'subsample': 0.9175968953970515, 'colsample_bytree': 0.6232027989094194, 'reg_alpha': 0.007761129300246636, 'reg_lambda': 0.013252537867683373}. Best is trial 0 with value: 2766.1486191860463.
[I 2026-11-14 23:37:19,395] Trial 1 finished with value: 2359.7325808502906 and parameters: {'n_estimators': 268, 'max_depth': 6, 'learning_rate': 0.012349896217470686, 'min_child_weight': 8, 'subsample': 0.6142179648660738, 'colsample_bytree': 0.7658563055858271, 'reg_alpha': 3.0245790870518783, 'reg_lambda': 0.27922331827407587}. Best is trial 1 with value: 2359.7325808502906.
[I 2026-11-14 23:37:19,782] Trial 2 finished with value: 2717.4280636809594 and parameters: {'n_estimators': 104, 'max_de